# Mint Tracker — run on Google Colab

Watches your wallets and posts a Discord alert when several of them MINT or BUY the same collection.

**One-time setup — add your keys to Colab Secrets (you never type them again):**

1. Click the **key icon (🔑)** in the left sidebar -> **Add new secret**.
2. Add each of these (Name must match exactly), and turn ON **Notebook access** for each:
   - `DISCORD_WEBHOOK_URL` — channel webhook (Channel Settings -> Integrations -> Webhooks -> New Webhook -> Copy URL)
   - `ALCHEMY_RPC_URL` — e.g. `https://robinhood-mainnet.g.alchemy.com/v2/YOUR_KEY`
   - `ALCHEMY_MAINNET_RPC_URL` — e.g. `https://eth-mainnet.g.alchemy.com/v2/YOUR_KEY`
   - `OPENSEA_API_KEY` — your OpenSea API key (real-time, OpenSea-only buys)
   - `DISCORD_ROLE_ID` — *(optional)* role ID to ping, or `everyone` / `here`

Secrets are stored on your Google account and persist across sessions — set them once. Then just run the cells in order.

> Colab stops after ~12h or on idle — for 24/7 use a VPS/Docker.

## 1. Code and dependencies

In [ ]:
REPO = "c8539-coder/main"
BRANCH = "claude/upbeat-knuth-1r9qh0"

import os
if not os.path.isdir("repo"):
    !git clone https://github.com/{REPO}.git repo
%cd repo
!git fetch origin && git checkout {BRANCH} && git pull
!pip -q install requests python-dotenv websocket-client
print("OK")

## 2. Upload the wallet list
Click the button and pick your `good_wallets.csv`.

In [ ]:
from google.colab import files
import os, shutil
os.makedirs("scripts/nft_top_wallets/out", exist_ok=True)
up = files.upload()
name = list(up)[0]
shutil.move(name, "scripts/nft_top_wallets/out/good_wallets.csv")
import csv
n = sum(1 for _ in csv.DictReader(open("scripts/nft_top_wallets/out/good_wallets.csv", encoding="utf-8")))
print(f"watchlist saved: {n} wallets")

## 3. Load settings (from Colab Secrets)
Reads your keys from Colab Secrets — nothing to type. If a secret is missing it will be listed below.

In [ ]:
import os
from google.colab import userdata

# keys pulled from Colab Secrets (🔑 icon). Set them once; they persist.
SECRETS = ["DISCORD_WEBHOOK_URL", "DISCORD_ROLE_ID", "ALCHEMY_RPC_URL",
           "ALCHEMY_MAINNET_RPC_URL", "OPENSEA_API_KEY"]
for key in SECRETS:
    try:
        val = userdata.get(key)
    except Exception:
        val = ""
    if val:
        os.environ[key] = val

# thresholds / window (not secret — edit freely)
os.environ["MINT_ALERT_MIN"]       = "5"    # quiet alert
os.environ["MINT_PING_MIN"]        = "15"   # role ping from this many wallets
os.environ["MINT_PING_STEP"]       = "15"   # re-ping every +N wallets
os.environ["MINT_WINDOW_SECONDS"]  = "900"  # real-time window (15 min)
os.environ["MINT_BACKFILL_BLOCKS"] = "0"    # no backfill on start

required = ["DISCORD_WEBHOOK_URL", "ALCHEMY_RPC_URL", "ALCHEMY_MAINNET_RPC_URL", "OPENSEA_API_KEY"]
missing = [k for k in required if not os.environ.get(k)]
print("missing secrets:", missing or "none — all set")

## 4. Run
This cell runs continuously and prints the log. Stop with the square button on the left. Keep the tab open.

Tip: run `!python -m scripts.mint_bot.main --test` first to post one sample alert and confirm the channel works.

In [ ]:
!python -m scripts.mint_bot.main